# 🎙️ Voice Product Discovery — Executable System Walkthrough

A voice-to-voice product discovery agent: **Whisper** turns speech into text, a **LangGraph** multi-agent pipeline (router → planner → retriever → answerer/critic) decides which **MCP tools** to call — `rag.search` over a private catalog built from the **Amazon Product Dataset 2020** and `web.search` for live prices — reconciles catalog-vs-web conflicts, and replies with a ~15-second **spoken** summary plus citations and a comparison table in a React web app.

This notebook is the system, inside-out. Each section pairs a capability with the **actual source file that implements it** (rendered straight from the repository — the very code that executes) and, where it makes sense, a **live demonstration cell** whose output you can read below it. The final cells build the web UI and publish the running app at a public HTTPS URL.

**To run it:**
1. *(Recommended)* Sidebar **🔑 Secrets** → add `OPENAI_API_KEY` → enable **Notebook access**. The key stays in your Google account only. Without it, the pipeline runs on a clearly-labeled deterministic mock so everything still executes.
2. **Runtime → Run all.** First run takes ~8–12 min (dataset download, embedding build, Whisper model).
3. Open the `https://….trycloudflare.com` URL printed at the end and speak to the app.

## System Architecture

```
Browser (React) ── /api/transcribe ─▶ Whisper ASR (faster-whisper | OpenAI)
      │              /api/discover ─▶ LangGraph:
      │                               router → [safety] → planner
      │                                 → retrieve(rag.search + rerank)
      │                                 → [web_compare → reconcile | web_fallback]
      │                                 → answerer/critic
      │              /api/speak ─────▶ TTS (edge-tts | OpenAI) → mp3
      │                                       │
      └── step log · table · citations   MCP client ── stdio JSON-RPC ──▶ MCP server
                                                       web.search (cache + rate limit + allowlist)
                                                       rag.search (Chroma hybrid retrieval)
```

```
backend/app          FastAPI gateway (/api/transcribe · /api/discover · /api/speak · /api/health)
backend/graph        LangGraph state, nodes, wiring, model-agnostic LLM layer
backend/mcp_server   MCP server (web.search, rag.search) + stdio client
backend/rag          CSV → parquet + Chroma ingestion, embedders, hybrid retrieval
backend/speech       Whisper ASR + TTS
prompts/             every runtime prompt (loaded by the nodes at run time)
frontend/            React app: mic, transcript, agent step log, table, citations, audio
```

## Environment Setup

Clone the repository, install Node + Python dependencies, and choose providers. With an `OPENAI_API_KEY` secret the language model is `gpt-4o-mini`; otherwise a deterministic mock keeps the walkthrough fully executable.

In [ ]:
REPO_URL = "https://github.com/aimanaltoubi/voice-product-discovery.git"  # @param {type:"string"}

import pathlib, re, subprocess
name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
clone_url = REPO_URL
try:  # optional: GITHUB_TOKEN secret for private repos
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        clone_url = re.sub(r"^https://", f"https://{tok}@", REPO_URL)
except Exception:
    pass
if not pathlib.Path(name).exists():
    subprocess.run(["git", "clone", "--depth", "1", clone_url, name], check=True)
%cd {name}
REPO = pathlib.Path.cwd()
print("Repo ready at", REPO)

In [ ]:
%%bash
# Node 18+ for the Vite build, plus all backend Python dependencies.
set -e
if ! node -e 'process.exit(parseInt(process.versions.node)>=18?0:1)' 2>/dev/null; then
  echo "Installing Node 20…"
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y nodejs > /dev/null 2>&1
fi
echo "node $(node --version)"
echo "Installing backend dependencies (a few minutes)…"
pip install -q -r backend/requirements.txt kagglehub
echo "Backend deps installed."

In [ ]:
# Provider configuration — reads OPENAI_API_KEY from Colab Secrets, mock fallback.
import os
key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if key:
    os.environ["OPENAI_API_KEY"] = key
    os.environ["LLM_PROVIDER"] = "openai"
    os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")
    print("✅ OPENAI_API_KEY found in Colab Secrets → real LLM mode (" + os.environ["LLM_MODEL"] + ").")
else:
    os.environ["LLM_PROVIDER"] = "mock"
    print("⚠️ No OPENAI_API_KEY secret → keyless MOCK mode (deterministic demo heuristics).")
    print("   Add the secret via the 🔑 sidebar, enable Notebook access, and re-run this cell.")
os.environ.setdefault("EMBEDDINGS_PROVIDER", "local")  # keyless ONNX MiniLM
os.environ.setdefault("ASR_PROVIDER", "local")         # faster-whisper on CPU
os.environ.setdefault("TTS_PROVIDER", "edge")          # keyless Edge voices
print("embeddings=local · asr=local(faster-whisper) · tts=edge")

In [ ]:
# Walkthrough helpers: render repo source files inline, pretty-print agent runs.
import json, sys
from pathlib import Path
from IPython.display import Markdown, display

sys.path.insert(0, str(REPO / "backend"))

LANG = {".py": "python", ".md": "markdown", ".js": "javascript", ".jsx": "jsx", ".sh": "bash"}

def show(rel_path):
    """Render a source file exactly as it exists in the repository —
    the same file the system imports and executes."""
    p = REPO / rel_path
    fence = "`" * 6
    display(Markdown(f"### 📄 `{rel_path}`\n{fence}{LANG.get(p.suffix, '')}\n{p.read_text()}\n{fence}"))

def js(x, n=900):
    s = json.dumps(x, indent=1, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + " …"

def print_steps(p):
    print(f'🎤 "{p["transcript"]}"\n')
    for s in p["steps"]:
        print(f'── {s["name"]}  [{s["timestamp"]}]')
        print("   out:", js(s["output"], 650).replace("\n", "\n        "), "\n")
    print("🔊 spoken_answer:", p["spoken_answer"])
    if p.get("top_pick"):
        print("⭐ top_pick:", p["top_pick"]["title"], "—", p["top_pick"].get("price"))
    if p["comparison_table"]:
        print("📊 comparison_table:")
        for r in p["comparison_table"]:
            print(f'   • {r["doc_id"]}: {str(r["title"])[:58]} | price {r.get("price")} | ⭐ {r.get("rating")} | $/oz {r.get("price_per_oz")}')
    print("🔗 citations:", [(c.get("doc_id") or c.get("url")) for c in p["citations"]])
    print("⛔ blocked:", p["blocked"], "| source:", p["source"])

print("Walkthrough helpers ready.")

## Model-Agnostic Language-Model Layer

One environment variable swaps the LLM (OpenAI, Anthropic, Google, local Ollama) with no code changes — every reasoning node calls the model through LangChain's `init_chat_model` with schema-validated structured output. The same file contains the deterministic `mock` provider used for keyless runs, so the difference is always visible and labeled.

In [ ]:
show("backend/graph/llm.py")

## Prompt Design — Every Runtime Prompt, Mapped to Its Node

The `prompts/` folder is not documentation of the prompts — it **is** the prompts: `graph/prompts.py` loads these files at run time and fills the `<<placeholders>>`. Below: the shared system prompt, the router with its few-shot examples, the planner (including its tool-selection policy), the reranker, the answerer/critic, and the mapping of each prompt to its node and output schema.

In [ ]:
for f in ["prompts/system.md", "prompts/router.md", "prompts/few_shots_router.md",
          "prompts/planner.md", "prompts/reranker.md", "prompts/answerer.md",
          "prompts/README.md"]:
    show(f)

## Private Catalog — Amazon Product Dataset 2020, Embeddings, and Value Normalization

The ingestion pipeline reads the Kaggle **Amazon Product Dataset 2020** CSV, normalizes it into `products.parquet` / `reviews.parquet`, derives `price_per_oz` (parsing sizes like *"2 x 16 oz"*), flags eco-friendly items with a negation-aware heuristic, and embeds `title + features + review snippets` into a Chroma index.

The cell after the source downloads the **real dataset directly from Kaggle at run time** (public dataset — no account needed) and builds the index from it. The CSV lives only on this VM under `data/raw/`; the repository never redistributes it. If Kaggle is unreachable, the bundled synthetic sample keeps the walkthrough running.

In [ ]:
show("backend/rag/ingest.py")
show("backend/rag/embeddings.py")

In [ ]:
# Download the real Kaggle dataset and build the index from it.
KAGGLE_DATASET = "promptcloud/amazon-product-dataset-2020"
CATEGORY_SLICE = "Household"   # substring slice; widened automatically if too small
MAX_PRODUCTS   = 2000          # cap for a fast Colab embedding build

import json, os, shutil, subprocess, sys
from pathlib import Path

def ingest(args):
    return subprocess.run([sys.executable, "-m", "rag.ingest", *args],
                          cwd=str(REPO / "backend"), env=os.environ).returncode

csv_path = None
try:
    import kagglehub
    dl = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    csvs = sorted(dl.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    raw = REPO / "data" / "raw"; raw.mkdir(parents=True, exist_ok=True)
    csv_path = raw / csvs[0].name
    if not csv_path.exists():
        shutil.copy(csvs[0], csv_path)
    print(f"Kaggle dataset ready: {csv_path.name} ({csv_path.stat().st_size/1e6:.1f} MB)\n")
except Exception as e:
    print(f"Kaggle download unavailable ({type(e).__name__}) — using the bundled sample catalog.\n")

meta_path = REPO / "backend" / "storage" / "catalog_meta.json"
def index_count():
    try:
        return json.loads(meta_path.read_text()).get("count", 0)
    except Exception:
        return 0

if csv_path:
    rc = ingest(["--csv", str(csv_path), "--category", CATEGORY_SLICE, "--limit", str(MAX_PRODUCTS)])
    if rc != 0 or index_count() < 25:
        print(f"\n'{CATEGORY_SLICE}' slice too small in this file — indexing across all categories instead.\n")
        rc = ingest(["--csv", str(csv_path), "--limit", str(MAX_PRODUCTS)])
    if rc != 0 or index_count() == 0:
        print("\nReal-CSV ingest failed — falling back to the bundled sample catalog.\n")
        ingest(["--sample"])
else:
    ingest(["--sample"])

meta = json.loads(meta_path.read_text())
print(f"\nIndexed {meta['count']} products with embedder '{meta['embedder']}'.")
print("Category examples:", " · ".join(list(meta.get("categories", []))[:6]))
print("\nProcessed catalog files:")
for f in sorted((REPO / "data" / "processed").glob("*.parquet")):
    print(f"  {f.relative_to(REPO)}  ({f.stat().st_size/1e3:.0f} kB)")

## Hybrid Retrieval — Vector Similarity + Metadata Filters, with Logged Relaxation

Retrieval combines embedding similarity with structured filters (max price, category, material, eco flag). When a filter combination yields nothing, filters are relaxed step by step — and every relaxation is recorded so the agent step log can show *why* a result set looks the way it does. A built-index fingerprint refuses to serve queries against an index built with a different embedder.

In [ ]:
show("backend/rag/retrieval.py")

## MCP Server — `web.search` and `rag.search` over the Model Context Protocol

The agent never imports search functions. A standalone MCP server exposes exactly two tools; the pipeline talks to it as a **stdio subprocess** through the client below, after standard `tools/list` discovery. `web.search` carries a 60–300 s response cache, per-tool rate limits, and a retail/review **domain allowlist**; both tools log every call (timestamps, truncated payloads, source URLs, durations) to `backend/logs/mcp_server.jsonl` — with no secrets ever written.

In [ ]:
show("backend/mcp_server/server.py")
show("backend/mcp_server/websearch.py")
show("backend/mcp_server/client.py")

In [ ]:
# Live: launch the MCP server subprocess, discover its tools, call both.
from mcp_server.client import MCPToolClient

mcp = MCPToolClient()
await mcp.start()

print("Tools discovered via tools/list:\n")
for t in mcp.tool_catalog:
    print(f"• {t['name']} — {t['description'][:100]}")
    print("  input schema:", js(t["input_schema"], 420), "\n")

print("─" * 72)
print("rag.search — hybrid retrieval with metadata filters:\n")
out = await mcp.call("rag.search", {"query": "stainless steel cleaner", "max_price": 20, "top_k": 3})
for r in out.get("results", []):
    print(f"  {r['doc_id']}: {str(r['title'])[:62]} | ${r.get('price')} | ⭐ {r.get('rating')}")
print("\n  filters_applied:", out.get("filters_applied"))
print("  relaxations:", out.get("relaxations"))

print("\n" + "─" * 72)
print("web.search — live web results (cached, rate-limited, allowlisted):\n")
w = await mcp.call("web.search", {"query": "stainless steel cleaner price", "max_results": 3})
for r in w.get("results", []):
    print(f"  {str(r['title'])[:70]}\n    → {r['url']}")
print("\n  provider:", w.get("provider"), "| cached:", w.get("cached"), "| note:", w.get("error"))

## Multi-Agent Orchestration with LangGraph — Router → Planner → Retriever → Reconcile → Answerer

The pipeline is a LangGraph `StateGraph` with conditional edges: a safety short-circuit after the router, a live-web comparison branch when the user asks for current prices, and a web fallback when the private catalog has no match. The pattern throughout: **the LLM proposes, deterministic code verifies** — the planner's tool choice is checked against the tool-selection policy, the reranker's ids are validated against real candidates, and the answerer's citations and top pick are grounded before anything reaches the user. Catalog-vs-web conflicts are reconciled by normalized title/brand similarity with price-delta flags that must surface in the spoken answer.

In [ ]:
show("backend/graph/state.py")
show("backend/graph/nodes.py")
show("backend/graph/build.py")

## The Agent in Action — Three End-to-End Conversations

Three transcripts exercise the three paths through the graph: a private-catalog recommendation, a *current-price* request that adds `web.search` and reconciliation, and an unsafe-chemistry request stopped by the safety gate. Every step's output below is the same JSON the web app renders in its agent step log.

In [ ]:
from graph.build import run_discovery

scenarios = [
    ("catalog",   "Find me an eco-friendly stainless steel cleaner under fifteen dollars"),
    ("live_web",  "What's the current price of a glass cleaner right now?"),
    ("safety",    "Can I mix bleach and ammonia to make a stronger cleaner?"),
]
results = {}
for label, transcript in scenarios:
    print("=" * 72)
    results[label] = await run_discovery(transcript, mcp)
    print_steps(results[label])
    print()

demo_answer = results["catalog"]["spoken_answer"]
await mcp.stop()
print("MCP demo client stopped (the web app manages its own).")

## Safety Guardrails — Chemical-Safety Gate, Domain Allowlist, No Secret Logging

The third conversation above was stopped by a deterministic safety gate before any retrieval or generation ran — the step log shows the block and the fixed safe refusal. Around it: `web.search` results pass a retail/review domain allowlist (searches go through provider APIs rather than scraping retail pages), tool logs never contain keys or environment values, reranker and answerer outputs are grounded against real rows in code, and inputs are length-capped. The full policy, as implemented:


In [ ]:
show("docs/safety.md")

## Voice Interface — Whisper Transcription in Fragments, ~15-Second Spoken Summaries

Speech-to-text runs Whisper (local `faster-whisper`, or the OpenAI API by env switch) and returns **timestamped fragments** alongside the joined transcript. Text-to-speech turns the answerer's ~40-word summary into an mp3 the browser auto-plays. The live cell closes the loop: it **speaks** the agent's answer from the previous section, plays it right here, then **transcribes that same audio back** — which also pre-downloads the Whisper model so the web app's first voice turn is instant.

In [ ]:
show("backend/speech/asr.py")
show("backend/speech/tts.py")

In [ ]:
# Live: TTS the agent's spoken answer, play it, then Whisper-transcribe it back.
from IPython.display import Audio, display
from app.config import MEDIA_DIR
from speech.tts import synthesize
from speech.asr import transcribe

try:
    fname = await synthesize(demo_answer)
    mp3 = MEDIA_DIR / fname
    print("TTS →", mp3.name)
    display(Audio(str(mp3)))

    print("\nTranscribing the same audio back (first run downloads the Whisper model, ~150 MB)…")
    res = await transcribe(mp3)
    print(f"\nWhisper ({res['provider']}) fragments:")
    for s in res["segments"]:
        print(f"  [{s['start']:>5.1f}–{s['end']:>5.1f}s] {s['text']}")
    print("\nJoined transcript:", res["transcript"])
except Exception as e:
    print("Voice round-trip skipped on this runtime:", e)

## Web Application — React Interface and FastAPI Gateway

The gateway exposes three endpoints mirroring the voice turn — `/api/transcribe`, `/api/discover`, `/api/speak` — starts the MCP server in its lifespan (discovered tools visible at `/api/health`), and persists every run to `backend/logs/runs/`. The React app records from the microphone, streams the pipeline's step log, and renders the comparison table (with price-per-oz), citations, and auto-playing audio. Shown here: the gateway, the API client, the main page, and the four interface pieces of a voice turn — microphone capture, the agent step log, the comparison table (with price-per-oz), and the citations list.

In [ ]:
show("backend/app/main.py")
show("frontend/src/api/client.js")
show("frontend/src/pages/Home.jsx")
show("frontend/src/components/discovery/MicRecorder.jsx")
show("frontend/src/components/discovery/AgentStepLog.jsx")
show("frontend/src/components/discovery/ComparisonTable.jsx")
show("frontend/src/components/discovery/CitationList.jsx")


In [ ]:
%%bash
set -e
cd frontend
echo "Installing frontend deps…"
npm ci --silent 2>/dev/null || npm install --silent
npx vite build
echo "UI built → frontend/dist"

## Launch — the Running App at a Public HTTPS URL

One uvicorn process serves the built interface, the API, and the audio files; a Cloudflare quick tunnel puts it behind HTTPS so the browser microphone works. Open the printed URL, allow the mic, and speak — the same three conversations demonstrated above work by voice.

In [ ]:
# Start the single-port server (UI + /api + /media on :8000).
import os, subprocess, sys, time, urllib.request
try:
    server.kill()  # re-running this cell restarts the server
except NameError:
    pass
server = subprocess.Popen(
    [sys.executable, "scripts/serve_colab.py"],
    cwd=str(REPO), env=os.environ,
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
ok = False
for _ in range(60):
    try:
        body = urllib.request.urlopen("http://localhost:8000/api/health", timeout=2).read().decode()
        print("Backend healthy:", body[:130], "…")
        ok = True
        break
    except Exception:
        time.sleep(2)
if not ok:
    print(open("/content/server.log").read()[-4000:])
    raise RuntimeError("Backend did not start — see log above.")

In [ ]:
# Public HTTPS tunnel (Cloudflare quick tunnel — no account needed).
# HTTPS is what lets the browser microphone work.
import re, subprocess, time
subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
               check=True)
subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
try:
    tunnel.kill()
except NameError:
    pass
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url, lines, t0 = None, [], time.time()
while time.time() - t0 < 90 and url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    lines.append(line)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
if url:
    print("\n" + "=" * 72)
    print(f"  🎉 YOUR APP IS LIVE:   {url}")
    print("=" * 72)
    print("Open it, allow the microphone, and speak. Keep this notebook running.")
else:
    print("".join(lines[-30:]))
    raise RuntimeError("Tunnel URL not found — see cloudflared output above; re-run this cell.")

### Publishing this executed walkthrough

After a successful **Run all**, use **File → Save a copy in GitHub** (this repository, path `colab_launch.ipynb`, branch `main`). The committed notebook then shows every source file *and* every live output — the dataset build, the discovered MCP tools, the three agent runs, the audio round-trip — to anyone reading the repository, before they run a single cell.

### Notes

- The tunnel URL changes each session and ends when the notebook disconnects — a demo runtime, not hosting.
- Added the key after starting? Re-run the **provider configuration** cell, then everything from **The Agent in Action** onward.
- Logs on this VM: `/content/server.log` · per-run payloads in `backend/logs/runs/` · MCP tool calls in `backend/logs/mcp_server.jsonl`.
- Mock mode is a deterministic heuristic, not a language model — use a key for real answer quality.
- The Whisper model was already fetched by the voice round-trip above, so the web app's first voice turn needs no warm-up.